In [1]:
import requests
import pandas as pd
import os
import re
from tqdm import tqdm #barra de progreso

session = requests.Session()

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) #para ver toda la tabla, en nuestro caso son solo 30 se supone que no tenemos que tener problema

API_KEY = '2c6d07b262d82febbd27cf5327a36d55' 

artistas = [
    "La Fuga",
    "Héroes del Silencio",
    "Billie Eilish",
    "Love of Lesbian",
    "Estopa",
    "Mägo de Oz",
    "Mr. Kilombo",
    "Rozalén",
    "Taburete",
    "Extremoduro",
    "La Plazuela",
    "Veintiuno",
    "Ojete Calor",
    "Rata Blanca",
    "Vetusta Morla",
    "Leiva",
    "Bad Bunny",
    "Enrique Bunbury",
    "Marea",
    "Joaquín Sabina",
    "Rosalía",
    "Queen",
    "The Lumineers",
    "Foo Fighters",
    "Muse",
    "Metallica",
    "Ginebras",
    "IZAL",
    "Kaiser Chiefs",
    "Residente"
]

artistas = [a.strip() for a in artistas]

def limpiar_bio(texto):
    """Elimina etiquetas HTML como <a href=...>Read more</a>"""
    return re.sub(r'<[^>]+>', '', texto).strip()

#Funcion para llamar a la API
def obtener_info_artista(nombre):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "artist.getinfo",
        "artist": nombre,
        "api_key": API_KEY,
        "format": "json",
        "autocorrect" : 1,  
        "lang": "es"
     }
    
    response = session.get(url, params=params, timeout=10)
    response.raise_for_status() #lanza excepción si hay error HTTP
    return response.json()

#Función para obtener los top tracks del artista con sus estadísticas
def obtener_top_tracks(nombre, limite=50):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "artist.getTopTracks",
        "artist": nombre,
        "api_key": API_KEY,
        "format": "json",
        "autocorrect": 1,
        "limit": limite #límite de canciones por artista
    }
    response = session.get(url, params=params, timeout=10)
    response.raise_for_status() #lanza excepción si hay error HTTP
    data = response.json()
    return [
        {
            "track": t["name"],
            "playcount": int(t["playcount"]),
            "listeners": int(t["listeners"])
        }
        for t in data.get("toptracks", {}).get("track", []) #devuelve nombre, playcount y listeners de cada canción
    ]

def procesar_artista(nombre):
    try:
        data = obtener_info_artista(nombre)
        artista = data["artist"]
        
        bio = limpiar_bio(artista["bio"]["content"])
        listeners = int(artista["stats"]["listeners"])
        playcount = int(artista["stats"]["playcount"])
        similares = [a["name"] for a in artista["similar"]["artist"]]
        tracks = obtener_top_tracks(nombre) #llamamos a la función de top tracks
        
        return {
            "artista": nombre,
            "biografia": bio,
            "listeners": listeners,
            "playcount": playcount,
            "similares": ", ".join(similares),
            "tracks": tracks #lista de diccionarios con track, playcount y listeners
        }
    
    except KeyError as e:
        print(f"[KeyError] '{nombre}': clave no encontrada {e}")  #except especifico que muestra qué clave faltó
        return None
    except session.RequestException as e:
        print(f"[RequestError] '{nombre}': {e}")
        return None

In [2]:
resultados = []
bar = tqdm(artistas, desc="Iniciando...")
 
for artista in bar:
    bar.set_description(f"Procesando: {artista}")
    info = procesar_artista(artista)
    if info:
        resultados.append(info)
 
print(f"\n✅ {len(resultados)}/{len(artistas)} artistas obtenidos correctamente")

session.close()

Procesando: Residente: 100%|██████████| 30/30 [00:13<00:00,  2.25it/s]         


✅ 30/30 artistas obtenidos correctamente


In [3]:
#Crear DataFrame principal con info de artistas
df_lastfm = pd.DataFrame(resultados).drop(columns=["tracks"]) #quitamos tracks para el df principal

# ── MEJORA: exportar a CSV para no tener que volver a llamar a la API ──
df_lastfm.to_csv("lastfm_artistas.csv", index=False, encoding="utf-8-sig")
print("✅ Datos guardados en lastfm_artistas.csv")

#Crear DataFrame separado para tracks con sus estadísticas
df_tracks = pd.DataFrame([
    {"artista": r["artista"], **t}
    for r in resultados
    for t in r["tracks"]
])
df_tracks.to_csv("lastfm_tracks.csv", index=False, encoding="utf-8-sig")
print("✅ Tracks guardados en lastfm_tracks.csv")

df_lastfm.head() #con esto veo 5

✅ Datos guardados en lastfm_artistas.csv
✅ Tracks guardados en lastfm_tracks.csv


,artista,biografia,listeners,playcount,similares
0,La Fuga,"La Fuga son Pedro (voz y guitarra), Nando (gui...",82666,2803586,"Marea, Platero y tú, Rosendo, Fito Y Fitipaldi..."
1,Héroes del Silencio,Héroes es el grupo de rock español de mayor éx...,202439,6113192,"Enrique Bunbury, Caifanes, Jaguares, Fobia, Ra..."
2,Billie Eilish,Billie Eilish Pirate Baird O'Connell​ (Los Áng...,4243656,839810873,"FINNEAS, Olivia Rodrigo, Melanie Martinez, Lan..."
3,Love of Lesbian,Love of Lesbian es un grupo de pop independien...,176649,11450422,"Lori Meyers, Viva Suecia, Izal, Vetusta Morla,..."
4,Estopa,Estopa es un grupo de pop/rumba/flamenco funda...,309416,8978193,"Melendi, El Canto del Loco, Extremoduro, Perez..."


In [ ]:
#para probar artista
procesar_artista('shakira')

In [ ]:
#merge con Deezer (cuando tengas df_deezer disponible)
df_final = df_deezer.merge(df_lastfm, on="artista", how="left")
df_final.head()